# Filtragem dos municípios cearenses

Este notebook seleciona as alternativas do estudo a partir das planilhas do SINISA 2025 (ano-base 2024). Entram:

- os municípios habilitados no Ranking ABES 2026, que têm os cinco indicadores;
- os municípios com **apenas um** indicador faltante.

O IRS3002 não conta como indicador faltante. Se ele estiver ausente, assume-se disposição final inadequada de resíduos, e o valor adaptado (100 − IRS3002, como no ABES) fica zero.

Para cada indicador, o notebook distingue um zero declarado de um dado que não existe. O dado que não existe pode ser um município fora da base, um indicador não calculado pelo SINISA, um município sem atendimento ou um município que não respondeu ao módulo de resíduos.

In [1]:
from pathlib import Path
import unicodedata

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SINISA = RAIZ / "dados" / "SINISA" / "2025"
ARQ_AGUA = SINISA / "abastecimento de água" / "Planilhas" / "Água - Base Municipal" / "SINISA_AGUA_Indicadores_Base Municipal_2024_Retificação.xlsx"
ARQ_ESGOTO = SINISA / "esgotamento sanitário" / "Planilhas" / "Esgoto - Base Municipal" / "SINISA_ESGOTO_Indicadores_Base Municipal_2024.xlsx"
ARQ_ESGOTO_GT = SINISA / "esgotamento sanitário" / "Planilhas" / "Esgoto - Base Municipal" / "SINISA_ESGOTO_Informações_Gestão Técnica de Esgoto_Base Municipal_2024.xlsx"
ARQ_RESIDUOS = SINISA / "resíduos sólidos" / "Planilhas" / "SINISA_RESIDUOS_Indicadores_2024.xlsx"
ARQ_ABES = RAIZ / "dados" / "ABES 2026" / "relatorio ABES 2026 CE.xlsx"
SAIDA = RAIZ / "resultados"

INDICADORES = ["IAG0001", "IES0001", "IES2003", "IRS0001", "IRS3002"]
# IRS3002 ausente vira zero (disposição inadequada) e não conta como faltante
CONTAM_COMO_FALTANTE = ["IAG0001", "IES0001", "IES2003", "IRS0001"]


def normaliza(nome):
    return unicodedata.normalize("NFC", str(nome)).strip().lower()

## 1. Leitura das planilhas

As planilhas de água e esgoto têm cabeçalho em várias linhas, e a linha com os códigos (`cod_IBGE`, `IAG0001` etc.) é usada como cabeçalho. A planilha de resíduos tem a coluna "Respondeu ao módulo de manejo de resíduos sólidos 2024" (`Sim`/`Não`), e os códigos ficam na linha 11.

In [2]:
agua = pd.read_excel(ARQ_AGUA, sheet_name="Indicadores de Gestão", header=11, dtype=object)
agua = agua[agua["UF"] == "CE"].set_index("cod_IBGE")

esgoto = pd.read_excel(ARQ_ESGOTO, sheet_name="Indicadores - Esgoto", header=10, dtype=object)
esgoto = esgoto[esgoto["UF"] == "CE"].set_index("cod_IBGE")

esgoto_gt = pd.read_excel(ARQ_ESGOTO_GT, header=10, dtype=object)
esgoto_gt = esgoto_gt[esgoto_gt["UF"] == "CE"].set_index("cod_IBGE")

bruto = pd.read_excel(ARQ_RESIDUOS, header=None, dtype=object)
residuos = bruto.iloc[13:].copy()
residuos.columns = bruto.iloc[10].astype(str).str.strip()
residuos = residuos.rename(columns={residuos.columns[0]: "Respondeu", residuos.columns[1]: "cod_IBGE",
                                    residuos.columns[2]: "Município", residuos.columns[3]: "UF"})
residuos = residuos[residuos["UF"] == "CE"].set_index("cod_IBGE")

abes = pd.read_excel(ARQ_ABES, dtype=str)
abes.columns = abes.columns.str.strip()

print(f"Água: {len(agua)} | Esgoto (indicadores): {len(esgoto)} | Esgoto (gestão técnica): {len(esgoto_gt)} | "
      f"Resíduos: {len(residuos)} | ABES: {len(abes)}")

Água: 182 | Esgoto (indicadores): 106 | Esgoto (gestão técnica): 171 | Resíduos: 184 | ABES: 184


## 2. Lista dos 184 municípios

A lista vem da planilha de resíduos, que tem todos os municípios do Ceará com o código do IBGE. A categoria do ABES é ligada pelo nome do município, porque o arquivo do ABES não traz o código.

In [3]:
mun = residuos[["Município"]].copy()
mun["chave"] = mun["Município"].map(normaliza)
abes["chave"] = abes["Município"].map(normaliza)
mun = mun.reset_index().merge(abes[["chave", "Porte", "Categoria", "Pontuação total"]], on="chave", how="left").set_index("cod_IBGE")
mun["Habilitado ABES"] = mun["Categoria"] != "Não habilitado"
assert mun["Categoria"].notna().all(), "município sem correspondência no ABES"
print(len(mun), "municípios;", mun["Habilitado ABES"].sum(), "habilitados no ABES")

184 municípios; 95 habilitados no ABES


## 3. Situação de cada indicador

Cada célula recebe o valor numérico, quando existe, e uma situação:

| Situação | Significado |
| --- | --- |
| `declarado` | valor numérico maior que zero |
| `zero declarado` | o prestador ou o município informou dados que resultam em zero |
| `não calculado` | o município está na base, mas o SINISA não calculou o indicador (ex.: população atendida maior que a total) |
| `sem atendimento` | esgoto: a Cagece tem a delegação, mas não presta o serviço (CAD2101 = "Com delegação sem atendimento") |
| `ausente da base` | o município não aparece na planilha, ou seja, não declarou |
| `não respondeu` | resíduos: coluna "Respondeu" = Não |

In [4]:
def classifica(valor, presente):
    if not presente:
        return None, "ausente da base"
    num = pd.to_numeric(valor, errors="coerce")
    if pd.isna(num):
        return None, "não calculado"
    return float(num), "zero declarado" if num == 0 else "declarado"


valores, situacao = {}, {}
for cod in mun.index:
    v, s = {}, {}
    v["IAG0001"], s["IAG0001"] = classifica(agua["IAG0001"].get(cod), cod in agua.index)
    for ind in ["IES0001", "IES2003"]:
        v[ind], s[ind] = classifica(esgoto[ind].get(cod), cod in esgoto.index)
        if s[ind] == "ausente da base" and cod in esgoto_gt.index \
                and "sem atendimento" in str(esgoto_gt.at[cod, "CAD2101"]).lower():
            s[ind] = "sem atendimento"
    respondeu = residuos.at[cod, "Respondeu"] == "Sim"
    for ind in ["IRS0001", "IRS3002"]:
        v[ind], s[ind] = classifica(residuos.at[cod, ind], respondeu)
        if not respondeu:
            s[ind] = "não respondeu"
    valores[cod], situacao[cod] = v, s

valores = pd.DataFrame.from_dict(valores, orient="index")[INDICADORES]
situacao = pd.DataFrame.from_dict(situacao, orient="index")[INDICADORES]

pd.concat({ind: situacao[ind].value_counts() for ind in INDICADORES}, axis=1).fillna(0).astype(int)

,IAG0001,IES0001,IES2003,IRS0001,IRS3002
declarado,180,106,94,161,143
não calculado,2,0,0,0,0
ausente da base,2,13,13,0,0
sem atendimento,0,65,65,0,0
zero declarado,0,0,12,0,18
não respondeu,0,0,0,23,23


## 4. Regras do ABES aplicadas aos valores

- **IRS3002 adaptado:** 100 − IRS3002, ou seja, a parcela com disposição final adequada. Se o IRS3002 estiver ausente, o valor adaptado é 0.
- **IES2003:** valores acima de 100 são limitados a 100, como no ABES.
- Valores negativos ou acima de 100 nos demais indicadores são sinalizados, porque o ABES exclui esses municípios.

In [5]:
matriz = valores.copy()
matriz["IRS3002"] = (100 - matriz["IRS3002"]).fillna(0)
matriz["IES2003"] = matriz["IES2003"].clip(upper=100)

fora_da_faixa = ((matriz[["IAG0001", "IES0001", "IRS0001", "IRS3002"]] < 0) |
                 (matriz[["IAG0001", "IES0001", "IRS0001", "IRS3002"]] > 100)).any(axis=1)
print("Municípios com valor fora de 0–100:", fora_da_faixa.sum())

Municípios com valor fora de 0–100: 0


## 5. Contagem de faltantes e filtragem

In [6]:
faltante = matriz[CONTAM_COMO_FALTANTE].isna()
mun["Nº faltantes"] = faltante.sum(axis=1)
mun["Indicador faltante"] = faltante.apply(lambda l: ", ".join(l.index[l]), axis=1)
mun["Situação do faltante"] = [
    ", ".join(situacao.at[cod, ind] for ind in CONTAM_COMO_FALTANTE if faltante.at[cod, ind])
    for cod in mun.index
]

pd.crosstab(mun["Habilitado ABES"], mun["Nº faltantes"], margins=True, margins_name="Total")

Nº faltantes,0,1,2,3,Total
Habilitado ABES,,,,,
False,0,11,62,16,89
True,95,0,0,0,95
Total,95,11,62,16,184


Conferência: todo habilitado no ABES deve ter zero faltantes, e a soma dos indicadores calculada aqui deve bater com a pontuação total do ABES.

In [7]:
hab = mun["Habilitado ABES"]
assert (mun.loc[hab, "Nº faltantes"] == 0).all()

pontuacao_abes = mun.loc[hab, "Pontuação total"].str.replace(",", ".").astype(float)
diferenca = (matriz.loc[hab].sum(axis=1) - pontuacao_abes).abs()
print(f"Maior diferença entre a soma calculada e a pontuação do ABES: {diferenca.max():.2f} ponto(s)")

Maior diferença entre a soma calculada e a pontuação do ABES: 0.00 ponto(s)


In [8]:
selecionados = mun[(mun["Nº faltantes"] <= 1) & ~fora_da_faixa]
resultado = selecionados[["Município", "Porte", "Categoria", "Nº faltantes", "Indicador faltante", "Situação do faltante"]] \
    .join(matriz).join(situacao.add_prefix("situação "))

print(f"{len(resultado)} municípios selecionados: "
      f"{(resultado['Nº faltantes'] == 0).sum()} completos e {(resultado['Nº faltantes'] == 1).sum()} com um faltante")
resultado[resultado["Nº faltantes"] == 1][["Município", "Categoria", "Indicador faltante", "Situação do faltante"]]

106 municípios selecionados: 95 completos e 11 com um faltante


,Município,Categoria,Indicador faltante,Situação do faltante
cod_IBGE,,,,
2301950,Barreira,Não habilitado,IRS0001,não respondeu
2302701,Campos Sales,Não habilitado,IRS0001,não respondeu
2303600,Catarina,Não habilitado,IRS0001,não respondeu
2304202,Crato,Não habilitado,IAG0001,não calculado
2304657,Graça,Não habilitado,IRS0001,não respondeu
2305001,Guaraciaba do Norte,Não habilitado,IRS0001,não respondeu
2305506,Iguatu,Não habilitado,IRS0001,não respondeu
2309300,Nova Russas,Não habilitado,IAG0001,não calculado
2310100,Palmácia,Não habilitado,IRS0001,não respondeu


## 6. Municípios que ficaram de fora

In [9]:
fora = mun[~mun.index.isin(resultado.index)]
fora.groupby(["Indicador faltante", "Situação do faltante"]).size().rename("municípios").to_frame()

municípios
Indicador faltante        Situação do faltante                                         
IAG0001, IES0001, IES2003 ausente da base, ausente da base, ausente da base           2
IES0001, IES2003          ausente da base, ausente da base                            9
                          sem atendimento, sem atendimento                           53
IES0001, IES2003, IRS0001 ausente da base, ausente da base, não respondeu             2
                          sem atendimento, sem atendimento, não respondeu            12

## 7. Exportação

O arquivo traz os valores dos cinco indicadores (o IRS3002 já adaptado) e a situação de cada um. O indicador faltante fica vazio, à espera da imputação.

In [10]:
SAIDA.mkdir(exist_ok=True)
resultado.to_csv(SAIDA / "municipios_selecionados.csv", encoding="utf-8-sig")
print("Salvo em", SAIDA / "municipios_selecionados.csv")

Salvo em /home/user/Rankeamento-Municipios-Saneamento/resultados/municipios_selecionados.csv
